# Impute your own dataset
## Goal
Upload a table with **samples in rows and numeric variables in columns**. Use the repository's existing nine-method workflow to optimize hyperparameters, compare reconstruction under MCAR/MAR/MNAR, select the method with the lowest validation RMSE, and impute your original dataset. Download the completed CSV with the original row order, column order, IDs, and metadata.

**Workflow preserved:** this notebook calls the existing `pipeline.tune()`, `pipeline.evaluate()`, original method adapters/search spaces, and `paper_seed_rmse_boxplots.py` without modifying them. The clinical-association stage is omitted. No CVD label or phenotype table is required.

**Reference requirement:** the existing reconstruction benchmark needs genuinely complete measurements. Use complete rows across all selected variables, or upload a separate complete reference table with the same variables and units. At least 50 complete reference rows are required by this companion. If none are available, the notebook explains the requirement and stops; it does not pre-impute unknown values and treat them as truth.

The final selected method is refitted on **all rows of your original dataset**, including its native missing values. Benchmark reference selection never removes rows or variables from the final export. Input values are used on their supplied scale, with the original methods' internal preprocessing unchanged.

## Setup
### 1. Clone the source repository
This is the same clone setup used by the TCGA notebook. Existing checkouts are reused. For a newer source version, start a fresh Colab runtime. Uploaded files and result folders are never automatically committed to GitHub.

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess

REPOSITORY_URL = 'https://github.com/niloofarmaani/proteomics-imputation-benchmark-cvd-association.git'
REPOSITORY_REF = 'main'
CLONE_DIR = (Path('/content') if Path('/content').is_dir() else Path.cwd()) / 'imputation-reproducibility'

# Reuse an existing checkout. In Colab, prefer the dedicated clone directory.
ROOT = next((p.resolve() for p in [CLONE_DIR, Path.cwd(), *Path.cwd().parents]
             if (p/'.git').exists() and (p/'reproducibility/pipeline.py').is_file()), None)
# Local readers can also run directly from an already extracted project folder.
if ROOT is None and not Path('/content').is_dir():
    ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
                 if (p/'reproducibility/pipeline.py').is_file()), None)

if ROOT is None:
    if shutil.which('git') is None:
        raise RuntimeError('Git is not installed. Install Git before running this cell.')
    if CLONE_DIR.exists():
        raise RuntimeError(f'{CLONE_DIR} already exists but is not a valid project checkout. Choose a different CLONE_DIR.')
    print('Cloning the source repository...')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPOSITORY_REF,
                    REPOSITORY_URL, str(CLONE_DIR)],
                   env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'}, check=True)
    ROOT = CLONE_DIR.resolve()

if not (ROOT/'reproducibility/pipeline.py').is_file():
    raise RuntimeError('The checkout does not contain the benchmark source modules.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Repository:', ROOT)
if (ROOT/'.git').exists():
    SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
    print('Source commit:', SOURCE_COMMIT)

### 2. Install the same Python and R dependencies
Uses the existing installer and original implementations: KNN, Iterative, MissForest, SoftImpute, AE, DAE, VAE, GAIN, and Transformer. Restart once if the installer reports that already-loaded packages changed, then rerun from the beginning.

In [ ]:
import subprocess
import importlib.metadata
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and os.environ.get('IMPUTATION_SKIP_INSTALL') != '1':
    loaded_before = {name: getattr(sys.modules.get(name), '__version__', None) for name in ['numpy', 'pandas', 'scipy', 'sklearn']}
    subprocess.run(['bash', str(ROOT/'scripts/install_colab.sh')], check=True)
    package_names = {'sklearn': 'scikit-learn'}
    changed = [name for name, version in loaded_before.items() if version is not None and version != importlib.metadata.version(package_names.get(name, name))]
    if changed:
        raise RuntimeError('Installed versions changed for already loaded packages: ' + ', '.join(changed) + '. Restart the runtime, then Run all again. Uploaded source files remain available.')
else:
    print('Local run: install requirements.txt and run Rscript scripts/install_r.R before continuing.')

### 3. Choose the existing experiment budget
`demo` keeps the existing quick budget: two trials per method, 30 neural epochs, and final seed 42. `paper_budget` keeps the existing method-specific trial counts, 300 epochs, and five final seeds. For five-seed plots with the quick tuning budget, explicitly replace `settings.final_seeds` below.

The tuning panel remains **MAR/MNAR × 10%/30%, seed 42**, with the original additional validation holdouts. Final reconstruction remains **MCAR/MAR/MNAR × 10%/30%/50%**. No new tuning objective, scaling procedure, search space, or imputation method is introduced. Full runs on large matrices can take substantial time.

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display
from reproducibility.pipeline import Settings, tune, evaluate
from reproducibility.methods import METHODS, PAPER_TRIALS, preflight
from reproducibility.user_data import (
    load_uploaded_data, reference_for_benchmark, initialize_uploaded_run,
    select_by_validation, impute_selected_dataset, plot_uploaded_benchmark,
    write_worked_example,
)

MODE = 'demo'  # 'demo' or 'paper_budget'
if MODE not in {'demo', 'paper_budget'}:
    raise ValueError('Choose demo or paper_budget.')
settings = Settings() if MODE == 'demo' else Settings.paper_budget()
# Optional quick-budget five-seed plots, with every other setting unchanged:
# from dataclasses import replace
# settings = replace(settings, final_seeds=(42, 100, 2024, 777, 999))

OUT = ROOT/'results'/pd.Timestamp.now(tz='UTC').strftime('own_data_%Y%m%dT%H%M%S%fZ')
print(preflight(list(settings.methods)))
print('Final seeds:', settings.final_seeds)
display(pd.DataFrame({'method': METHODS, 'trials': [PAPER_TRIALS[m] if MODE == 'paper_budget' else settings.trials for m in METHODS]}))

## Steps
### 4. Upload your dataset
CSV and TSV are supported. Use one header row. Empty cells, `NA`, `N/A`, `NaN`, `null`, and `None` are treated as missing **in numeric variables**. Numeric values are not rescaled or encoded by this notebook. Put measurements on a scientifically meaningful common scale before upload because RMSE depends on the supplied units.

- `ID_COLUMN='auto'` recognizes a single `eid`, `sample_id`, `patient_id`, `id`, or `Unnamed: 0` column. Set a column name explicitly when needed, or `None` when there is no ID column.
- Put labels, outcomes, batch information, and other non-imputed columns in `METADATA_COLUMNS`, for example `['label']`. They are kept in the exported CSV and never passed to the imputers.
- All other columns must be numeric variables. Their full names and ordering are preserved. A fully missing variable cannot be estimated by these original adapters and is reported as an input error.

Keep `INPUT_MODE='upload'` to use your data in Colab. `example` generates an explicitly synthetic test dataset; `path` reads an existing local file. The example is for checking execution, not a scientific result.

In [ ]:
INPUT_MODE = os.environ.get('IMPUTATION_INPUT_MODE', 'upload')  # 'upload', 'path', 'example'
INPUT_PATH = ''  # For INPUT_MODE='path', enter the absolute CSV/TSV path.
ID_COLUMN = 'auto'
METADATA_COLUMNS = []  # Example: ['label', 'batch']; these columns are never imputed.
CSV_SEPARATOR = None   # None = comma for CSV, tab for TSV; or set ',' / '\t'.

UPLOAD_DIR = ROOT/'data/user_uploads'
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
if INPUT_MODE == 'example':
    input_path = write_worked_example(UPLOAD_DIR/'synthetic_own_data_example.csv')
    print('Using synthetic execution example.')
elif INPUT_MODE == 'path':
    input_path = Path(INPUT_PATH).expanduser().resolve()
elif INPUT_MODE == 'upload':
    if not IN_COLAB:
        raise RuntimeError("For local use, set INPUT_MODE='path' and INPUT_PATH, or choose 'example'.")
    from google.colab import files
    os.chdir(UPLOAD_DIR)
    try:
        uploaded = files.upload()
    finally:
        os.chdir(ROOT)
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one input CSV/TSV.')
    input_path = UPLOAD_DIR/Path(next(iter(uploaded))).name
else:
    raise ValueError('Choose upload, path, or example.')

data = load_uploaded_data(input_path, id_column=ID_COLUMN,
                          metadata_columns=METADATA_COLUMNS, separator=CSV_SEPARATOR)
print('Input samples:', len(data.matrix), '| numeric variables:', len(data.feature_columns))
print('ID column:', data.id_column, '| preserved metadata:', data.metadata_columns)
print('Native missing measurements:', int(data.matrix.isna().sum().sum()))
display(pd.DataFrame({'variable': data.feature_columns,
                      'missing_cells': data.matrix.isna().sum().to_numpy(),
                      'missing_fraction': data.matrix.isna().mean().to_numpy()}).head(30))

### 5. Provide complete reference values for the unchanged benchmark
Default `complete_rows` uses every fully observed row across the same selected variables. This does not search for a smaller variable panel and does not pre-impute the reference. At least 50 such rows are required.

If your input lacks enough complete rows, use `upload_reference` or `reference_path`. The separate table must have the **same variable names, measurement units, and preprocessing**. Its variable order can differ because names are matched explicitly. Set its own ID/metadata settings below. The reference must be complete. No hidden/native missing measurement is ever treated as known truth.

The reference determines which dataset the method ranking applies to. Complete-case selection can be unrepresentative of the incomplete rows. A separate reference may differ from your target data. The resulting winner is therefore the best validation performer on this reference and budget, not a guarantee about unknown values.

In [ ]:
REFERENCE_MODE = 'complete_rows'  # 'complete_rows', 'upload_reference', or 'reference_path'
REFERENCE_PATH = ''
REFERENCE_ID_COLUMN = 'auto'
REFERENCE_METADATA_COLUMNS = []
REFERENCE_SEPARATOR = None
reference_data = None

if REFERENCE_MODE == 'upload_reference':
    if not IN_COLAB:
        raise RuntimeError("Use REFERENCE_MODE='reference_path' locally.")
    from google.colab import files
    reference_dir = UPLOAD_DIR/'reference'
    reference_dir.mkdir(exist_ok=True)
    os.chdir(reference_dir)
    try:
        uploaded_reference = files.upload()
    finally:
        os.chdir(ROOT)
    if len(uploaded_reference) != 1:
        raise ValueError('Upload exactly one complete reference CSV/TSV.')
    reference_path = reference_dir/Path(next(iter(uploaded_reference))).name
elif REFERENCE_MODE == 'reference_path':
    reference_path = Path(REFERENCE_PATH).expanduser().resolve()
elif REFERENCE_MODE != 'complete_rows':
    raise ValueError('Choose a documented reference mode.')

if REFERENCE_MODE != 'complete_rows':
    reference_data = load_uploaded_data(reference_path, id_column=REFERENCE_ID_COLUMN,
                                      metadata_columns=REFERENCE_METADATA_COLUMNS,
                                      separator=REFERENCE_SEPARATOR)
reference, input_audit = reference_for_benchmark(data, reference_data)
display(pd.Series(input_audit, name='Input/reference audit'))
OUT = initialize_uploaded_run(data, reference, settings, OUT, input_audit, reference_data)
print('Run directory:', OUT)

### 6. Inspect the original simulation and validation masks
Calls the existing missingness and validation-mask functions. All methods get identical masks. Validation holdouts are disjoint from simulated test omissions. Rank-block MAR/MNAR retain the documented meaning and limitations of the original public workflow; they do not identify your input's real missingness mechanism.

In [ ]:
from reproducibility.missingness import corrupt, validation_mask, combo_seed
preview, simulated_mask = corrupt(reference, 'MAR', .3, settings.tuning_seed)
heldout_mask = validation_mask(preview, combo_seed(settings.tuning_seed, 'MAR', .3))
assert not (simulated_mask & heldout_mask).any()
display(pd.DataFrame({'cells': [int(simulated_mask.sum()), int(heldout_mask.sum())]},
                     index=['Original simulated test omissions', 'Additional tuning holdouts']))

### 7. Optimize hyperparameters and freeze one configuration per method
This is the existing `tune()` call. Its objective remains mean validation RMSE across the original four tuning conditions. The same implementations, search spaces, internal early stopping/refit, and failure behavior are retained. A failed method stops the workflow instead of silently selecting from a smaller set.

The lowest frozen validation RMSE selects the method to refit later. That choice is recorded **before** final reconstruction is evaluated. Ties follow the configured method order.

In [ ]:
frozen = tune(reference, OUT, settings)
selected_method, validation_ranking = select_by_validation(frozen, settings.methods)
validation_ranking.to_csv(OUT/'method_selection.csv', index=False)
(OUT/'selected_method_before_final_evaluation.json').write_text(json.dumps({
    'method': selected_method,
    'selection': 'lowest frozen validation RMSE before final reconstruction evaluation',
}, indent=2))
trials = pd.read_csv(OUT/'validation_trials.csv')
display(trials[['method', 'trial', 'state', 'objective']].round(5))
display(validation_ranking.round(5))
print('Validation-selected method:', selected_method)
display(pd.DataFrame([{'method': method, 'frozen_parameters': json.dumps(item['params'])}
                      for method, item in frozen.items()]))

### 8. Compare reconstruction with the existing frozen-parameter evaluation
This is the unchanged `evaluate()` call. It scores RMSE only at simulated missing measurements in the complete reference. The original 10%/30%/50% MCAR/MAR/MNAR comparison is retained. Final benchmark scores are reported for comparison and do not replace the validation-selected winner.

The supplied method named SoftImpute retains the original rank-truncated SVD implementation, including its documented difference from singular-value-shrinkage SoftImpute.

In [ ]:
reconstruction = evaluate(reference, OUT, settings, frozen)
comparison = reconstruction.groupby(['method', 'mechanism', 'fraction']).rmse.mean().unstack(['mechanism', 'fraction'])
display(comparison.round(5))
print('Final scores describe the complete reference, not errors at unknown native missing cells.')

### 9. Generate the original paper-style comparison plot
Uses `legacy/Evaluation/paper_seed_rmse_boxplots.py`, with the same colours, method order, grouped missingness fractions, and stacked mechanism panels. Dataset labels and RMSE units refer to your uploaded reference.

With one seed, collapsed boxes show individual scores and provide no replicate uncertainty estimate. Five seeds produce IQR boxes, median lines, 1.5-IQR whiskers, and individual seed points. PDF, 300-dpi PNG, per-seed values, and summaries are saved.

In [ ]:
plot_outputs, plotted_values, reconstruction_summary = plot_uploaded_benchmark(reconstruction, OUT, settings)
print('PDF:', plot_outputs['main_all_conditions_figure']['pdf'])
print('PNG:', plot_outputs['main_all_conditions_figure']['png'])
display(reconstruction_summary[['missingness', 'fraction', 'method', 'count', 'mean', 'std']].round(5))

### 10. Refit the selected method on your original dataset
The original method adapter now receives **all original rows and selected numeric variables**, with their actual native missing values and the frozen hyperparameters. IDs and metadata stay outside the imputer. No unknown reference values or clinical labels are supplied to fitting.

The export restores original column names and order, keeps sample IDs (including leading zeros), preserves observed measurement strings and metadata, and fills only originally missing numeric measurements. The completed dataset is not one of the artificially corrupted benchmark matrices. If your input was already complete, the export is unchanged and no unnecessary final fit is claimed.

In [ ]:
best_csv, best_imputed = impute_selected_dataset(data, OUT, settings, frozen, selected_method)
print('Selected original method:', selected_method)
print('Original samples retained:', len(best_imputed))
print('Originally missing measurements filled:', input_audit['native_missing_cells'])
print('Completed dataset:', best_csv)
display(best_imputed.head())
display(pd.Series(json.loads((OUT/'best_method.json').read_text()), name='Selected method and export'))
# Show actual filled cells, including samples outside the complete reference.
changed_cells = []
for i, j in np.argwhere(data.matrix.isna().to_numpy())[:12]:
    column = data.feature_columns[j]
    changed_cells.append({'sample': data.original.iloc[i][data.id_column] if data.id_column else int(i) + 1,
                          'variable': column, 'before': data.original.iloc[i][column],
                          'imputed': best_imputed.iloc[i][column]})
if changed_cells:
    display(pd.DataFrame(changed_cells))


## Checks
### 11. Verify the comparison and export
Checks cover the full method/mechanism/fraction/seed grid, finite scores, selection based only on the frozen validation results, complete numeric output, and preservation of observed values, IDs, and metadata. The original reference benchmark and final original-data imputation have separate output files.

In [ ]:
expected = len(settings.methods)*len(settings.mechanisms)*len(settings.fractions)*len(settings.final_seeds)
assert len(reconstruction) == expected
assert not reconstruction.duplicated(['method', 'mechanism', 'fraction', 'seed']).any()
assert np.isfinite(reconstruction.rmse).all()
assert select_by_validation(frozen, settings.methods)[0] == selected_method
assert best_imputed.columns.equals(data.original.columns)
assert len(best_imputed) == len(data.original)
for j, column in enumerate(data.feature_columns):
    originally_observed = data.matrix.iloc[:, j].notna().to_numpy()
    assert best_imputed.loc[originally_observed, column].equals(data.original.loc[originally_observed, column])
    assert np.isfinite(pd.to_numeric(best_imputed[column]).to_numpy()).all()
for column in [c for c in data.original if c not in data.feature_columns]:
    assert best_imputed[column].equals(data.original[column])
print(f'Complete reconstruction grid: {len(reconstruction)}/{expected}')
print('All original rows, variable ordering, observed measurements, IDs and metadata preserved.')
print('Every originally missing numeric measurement is filled.')

## Next Steps
### 12. Download the best method's imputed CSV
This file contains your original table with only missing numeric measurements filled. Excluded metadata columns can still contain their original missing entries. The download runs in Colab; local users can open the printed output path.

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(best_csv))
else:
    print('Best imputed CSV:', best_csv)

### 13. Keep the complete results bundle
The archive includes your input table, reference matrix, masks, Optuna study, tuning tables, frozen parameters, reconstruction comparisons, figures, selected-method record, logs, environment, and completed CSV. It is created in your runtime and is not uploaded to GitHub. Set `DOWNLOAD_FULL_BUNDLE=True` to download it as well.

In [ ]:
DOWNLOAD_FULL_BUNDLE = False
archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT)
print('Results bundle:', archive)
if IN_COLAB and DOWNLOAD_FULL_BUNDLE:
    from google.colab import files
    files.download(archive)